In [2]:

!pip -q install pandas nltk scikit-learn gradio


In [3]:

import pandas as pd

faqs = [
    {"question": "What is artificial intelligence?",
     "answer": "Artificial intelligence is technology that enables computers to perform tasks that normally require human intelligence."},
    {"question": "What is machine learning?",
     "answer": "Machine learning is a branch of AI that enables computers to learn patterns from data."},
    {"question": "What is the internet?",
     "answer": "The internet is a global network that connects computers and devices."},
    {"question": "What is a chatbot?",
     "answer": "A chatbot is a computer program that interacts with users through text or voice."},
    {"question": "What is Python?",
     "answer": "Python is a popular programming language known for its readable syntax."},
    {"question": "What is programming?",
     "answer": "Programming is the process of writing instructions for a computer to perform tasks."},
    {"question": "What is a database?",
     "answer": "A database is an organized collection of data that can be stored and retrieved."},
    {"question": "What is cloud computing?",
     "answer": "Cloud computing provides computing services such as storage and servers over the internet."},
    {"question": "What is cybersecurity?",
     "answer": "Cybersecurity is the practice of protecting systems, networks, and data from digital attacks."},
    {"question": "What is data science?",
     "answer": "Data science uses data analysis, statistics, and computing to discover useful insights."},
    {"question": "What is a smartphone?",
     "answer": "A smartphone is a mobile device that provides calling, internet access, and applications."},
    {"question": "What is an operating system?",
     "answer": "An operating system manages computer hardware and software resources."},
    {"question": "What is a web browser?",
     "answer": "A web browser is software used to access and view websites."},
    {"question": "What is email?",
     "answer": "Email is a method of sending and receiving digital messages over a network."},
    {"question": "What is a search engine?",
     "answer": "A search engine helps users find information on the internet."},
    {"question": "What is a computer virus?",
     "answer": "A computer virus is malicious software that can spread and disrupt computer operations."},
    {"question": "How can I create a strong password?",
     "answer": "Use a long, unique password with a mix of character types and enable multi-factor authentication when possible."},
    {"question": "What is social media?",
     "answer": "Social media consists of online platforms where people share content and communicate."},
    {"question": "What is e-learning?",
     "answer": "E-learning is learning through digital devices and online educational resources."},
    {"question": "What is renewable energy?",
     "answer": "Renewable energy comes from naturally replenished sources such as sunlight and wind."},
    {"question": "What is a robot?",
     "answer": "A robot is a machine that can carry out tasks automatically or through programmed instructions."},
    {"question": "What is HTML?",
     "answer": "HTML is a markup language used to structure content on web pages."},
    {"question": "What is an algorithm?",
     "answer": "An algorithm is a sequence of steps used to solve a problem or complete a task."},
    {"question": "What is a mobile application?",
     "answer": "A mobile application is software designed to run on a smartphone or tablet."},
    {"question": "How can I stay safe online?",
     "answer": "Use strong passwords, enable multi-factor authentication, update your software, and avoid suspicious links."}
]

df = pd.DataFrame(faqs)
df.to_csv("general_faqs.csv", index=False)

print("FAQ dataset created successfully!")
print("Total questions:", len(df))
display(df.head(10))

FAQ dataset created successfully!
Total questions: 25


,question,answer
0,What is artificial intelligence?,Artificial intelligence is technology that ena...
1,What is machine learning?,Machine learning is a branch of AI that enable...
2,What is the internet?,The internet is a global network that connects...
3,What is a chatbot?,A chatbot is a computer program that interacts...
4,What is Python?,Python is a popular programming language known...
5,What is programming?,Programming is the process of writing instruct...
6,What is a database?,A database is an organized collection of data ...
7,What is cloud computing?,Cloud computing provides computing services su...
8,What is cybersecurity?,Cybersecurity is the practice of protecting sy...
9,What is data science?,"Data science uses data analysis, statistics, a..."


In [4]:

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()

    tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

df["processed_question"] = df["question"].apply(preprocess)

print("Text preprocessing completed!")
display(df[["question", "processed_question"]].head())







Text preprocessing completed!


,question,processed_question
0,What is artificial intelligence?,artifici intellig
1,What is machine learning?,machin learn
2,What is the internet?,internet
3,What is a chatbot?,chatbot
4,What is Python?,python


In [5]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
faq_vectors = vectorizer.fit_transform(df["processed_question"])

def get_answer(user_question):
    cleaned_question = preprocess(user_question)

    if not cleaned_question.strip():
        return "Please enter a question.", 0.0

    user_vector = vectorizer.transform([cleaned_question])
    scores = cosine_similarity(user_vector, faq_vectors)[0]

    best_index = scores.argmax()
    best_score = float(scores[best_index])

    if best_score < 0.15:
        return (
            "Sorry, I couldn't find a suitable answer. "
            "Please try asking about AI, Python, computers, "
            "the internet, or other topics in my FAQ list.",
            best_score
        )

    answer = df.iloc[best_index]["answer"]
    return answer, best_score

print("Chatbot matching system is ready!")

Chatbot matching system is ready!


In [6]:

test_questions = [
    "Explain AI",
    "Tell me about the Python language",
    "How do I protect my account online?",
    "What is a database?",
    "What is the weather on Mars?"
]

for question in test_questions:
    answer, score = get_answer(question)

    print("You:", question)
    print("Bot:", answer)
    print("Similarity score:", round(score, 3))
    print("-" * 50)


You: Explain AI
Bot: Sorry, I couldn't find a suitable answer. Please try asking about AI, Python, computers, the internet, or other topics in my FAQ list.
Similarity score: 0.0
--------------------------------------------------
You: Tell me about the Python language
Bot: Python is a popular programming language known for its readable syntax.
Similarity score: 1.0
--------------------------------------------------
You: How do I protect my account online?
Bot: Use strong passwords, enable multi-factor authentication, update your software, and avoid suspicious links.
Similarity score: 0.577
--------------------------------------------------
You: What is a database?
Bot: A database is an organized collection of data that can be stored and retrieved.
Similarity score: 1.0
--------------------------------------------------
You: What is the weather on Mars?
Bot: Sorry, I couldn't find a suitable answer. Please try asking about AI, Python, computers, the internet, or other topics in my FAQ li

In [7]:

from google.colab import files
files.download("general_faqs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download the English stopwords list
nltk.download("stopwords", quiet=True)

# Prepare NLP tools
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

# Function to clean text
def preprocess(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"[^a-z0-9\s]", " ", text)  # Remove punctuation
    tokens = text.split()  # Split into words

    # Remove common words and apply stemming
    tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

# Apply preprocessing to every FAQ question
df["processed_question"] = df["question"].apply(preprocess)

print("Text preprocessing completed successfully!")
display(df[["question", "processed_question"]].head(10))



Text preprocessing completed successfully!


,question,processed_question
0,What is artificial intelligence?,artifici intellig
1,What is machine learning?,machin learn
2,What is the internet?,internet
3,What is a chatbot?,chatbot
4,What is Python?,python
5,What is programming?,program
6,What is a database?,databas
7,What is cloud computing?,cloud comput
8,What is cybersecurity?,cybersecur
9,What is data science?,data scienc


In [10]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Convert all FAQ questions into numerical vectors
faq_vectors = vectorizer.fit_transform(
    df["processed_question"]
)

print("TF-IDF conversion completed successfully!")
print("Number of FAQ questions:", faq_vectors.shape[0])
print("Number of text features:", faq_vectors.shape[1])


TF-IDF conversion completed successfully!
Number of FAQ questions: 25
Number of text features: 38


In [11]:

from sklearn.metrics.pairwise import cosine_similarity

def get_answer(user_question):

    # Clean the user's question
    cleaned_question = preprocess(user_question)

    # Handle empty input
    if not cleaned_question.strip():
        return "Please enter a question.", 0.0

    # Convert the user's question into a TF-IDF vector
    user_vector = vectorizer.transform([cleaned_question])

    # Compare it with all stored FAQ questions
    scores = cosine_similarity(user_vector, faq_vectors)[0]

    # Find the FAQ with the highest similarity
    best_index = scores.argmax()
    best_score = float(scores[best_index])

    # Reject weak matches
    if best_score < 0.15:
        return (
            "Sorry, I couldn't find a suitable answer. "
            "Please try another question related to my FAQs.",
            best_score
        )

    # Retrieve the matching answer
    answer = df.iloc[best_index]["answer"]

    return answer, best_score

print("Chatbot matching system is ready!")

Chatbot matching system is ready!


In [13]:

test_questions = [
    "Explain artificial intelligence",
    "Tell me about Python",
    "What is a database?",
    "How can I stay safe on the internet?",
    "What is the weather on Mars?"
]

for question in test_questions:
    answer, score = get_answer(question)

    print("User:", question)
    print("Bot:", answer)
    print("Similarity score:", round(score, 3))
    print("-" * 45)


User: Explain artificial intelligence
Bot: Artificial intelligence is technology that enables computers to perform tasks that normally require human intelligence.
Similarity score: 1.0
---------------------------------------------
User: Tell me about Python
Bot: Python is a popular programming language known for its readable syntax.
Similarity score: 1.0
---------------------------------------------
User: What is a database?
Bot: A database is an organized collection of data that can be stored and retrieved.
Similarity score: 1.0
---------------------------------------------
User: How can I stay safe on the internet?
Bot: Use strong passwords, enable multi-factor authentication, update your software, and avoid suspicious links.
Similarity score: 0.667
---------------------------------------------
User: What is the weather on Mars?
Bot: Sorry, I couldn't find a suitable answer. Please try another question related to my FAQs.
Similarity score: 0.0
----------------------------------------

In [14]:

import gradio as gr

def chat(message, history):
    answer, score = get_answer(message)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="💬 General FAQ Chatbot",
    description="Ask me questions about technology, computers, programming, and the internet.",
    examples=[
        "What is artificial intelligence?",
        "What is Python?",
        "What is cybersecurity?",
        "How can I stay safe online?"
    ]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://97d250298499fe2356.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:

test_questions = [
    "What is artificial intelligence?",
    "Explain AI",
    "Tell me about Python",
    "What is a database?",
    "How do I protect myself online?",
    "What is the weather on Mars?"
]

for question in test_questions:
    answer, score = get_answer(question)

    print("User:", question)
    print("Chatbot:", answer)
    print("Similarity score:", round(score, 3))
    print("-" * 60)



User: What is artificial intelligence?
Chatbot: Artificial intelligence is technology that enables computers to perform tasks that normally require human intelligence.
Similarity score: 1.0
------------------------------------------------------------
User: Explain AI
Chatbot: Sorry, I couldn't find a suitable answer. Please try another question related to my FAQs.
Similarity score: 0.0
------------------------------------------------------------
User: Tell me about Python
Chatbot: Python is a popular programming language known for its readable syntax.
Similarity score: 1.0
------------------------------------------------------------
User: What is a database?
Chatbot: A database is an organized collection of data that can be stored and retrieved.
Similarity score: 1.0
------------------------------------------------------------
User: How do I protect myself online?
Chatbot: Use strong passwords, enable multi-factor authentication, update your software, and avoid suspicious links.
Simila

In [16]:

from google.colab import files

df.to_csv("general_faqs.csv", index=False)
files.download("general_faqs.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>